# Chapter 4: Hydrostatics — Restoring Forces, Ballast, and Natural Periods

In Chapter 3, we analyzed the kinetic forces that resist acceleration and velocity (Mass, Coriolis, Damping). In **Chapter 4**, we turn our attention to **Hydrostatics**. 

Hydrostatics dictates the forces and moments that act to restore a vessel to its equilibrium position when tilted, displaced, or weighted down. In Fossen’s master equation, this is captured by the vector **$g(\eta)$**:

$$M \dot{\nu} + C(\nu)\nu + D(\nu)\nu + g(\eta) = \tau$$

For a surface vessel or a submerged vehicle, $g(\eta)$ acts like a non-linear mechanical spring. If your guidance and control algorithms do not accurately account for $g(\eta)$, the vessel will fail to maintain trim, list heavily during basic maneuvers, or in worst-case scenarios, undergo stability failure (capsizing).

---

## 1. The Righting Arm ($GZ$) and Metacentric Height ($GM$)

For a surface vessel, stability is governed by the relative positions of three points along the centerline:
* **$G$ (Center of Gravity):** Where the total weight of the vessel acts downward.
* **$B$ (Center of Buoyancy):** The geometric center of the displaced water volume, acting upward.
* **$M$ (Metacenter):** The intersection of the vertical buoyancy line through the tilted hull with the original centerline.



The distance between $G$ and $M$ is the **Metacentric Height ($GM$)**. When the ship heels over by an angle $\phi$, the horizontal distance between the gravity and buoyancy vectors creates a righting lever arm, $GZ \approx GM \sin(\phi)$. 

The restoring moment in Roll is directly proportional to this distance:
$$\tau_{\text{roll}} = -mg \cdot GM \sin(\phi)$$

Let's look at how changes in cargo or ballast loading affect this righting lever and stability margin.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def simulate_stability_curve(vessel_weight_tonnes, kg_meters):
    """
    Plots the Righting Arm (GZ) curve as a function of heel angle.
    Shows how changing the height of the Center of Gravity (KG) can lead to 
    stable (stiff/tender) or unstable (capsizing) hull profiles.
    """
    # Fixed hull geometry characteristics
    m = vessel_weight_tonnes * 1000.0  # kg
    g = 9.81
    weight_force = m * g
    
    # Transverse Metacenter height above keel (KM) - assumed fixed for this draft hull profile
    km_meters = 6.5 
    
    # Calculate Metacentric Height: GM = KM - KG
    gm = km_meters - kg_meters
    
    # Generate heel angles from 0 to 60 degrees
    angles_deg = np.linspace(0, 60, 100)
    angles_rad = np.radians(angles_deg)
    
    # Standard wall-sided hull approximation for the GZ curve: GZ = (GM + 0.5*BM*tan^2(phi)) * sin(phi)
    # Simplifying for visualization of primary GM impact:
    gz_curve = gm * np.sin(angles_rad) + 0.5 * 2.0 * (np.tan(angles_rad)**2) * np.sin(angles_rad)
    
    # Calculate restoring torque in kN·m
    restoring_torque_knm = (weight_force * gz_curve) / 1000.0

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    
    # Plot 1: GZ Lever Arm
    ax1.plot(angles_deg, gz_curve, color='royalblue', linewidth=3, label='GZ Lever Arm')
    ax1.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax1.set_title(f"Righting Arm (GZ Curve) | GM = {gm:.2f}m", fontsize=11)
    ax1.set_xlabel("Heel Angle (Degrees)", fontsize=10)
    ax1.set_ylabel("GZ Lever Arm (Meters)", fontsize=10)
    ax1.grid(True, linestyle=':', alpha=0.6)
    
    # Plot 2: Actual Restoring Torque
    ax2.plot(angles_deg, restoring_torque_knm, color='crimson', linewidth=3, label='Restoring Moment')
    ax2.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax2.set_title("Hydrostatic Restoring Torque ($\tau_{roll}$)", fontsize=11)
    ax2.set_xlabel("Heel Angle (Degrees)", fontsize=10)
    ax2.set_ylabel("Torque (kN·m)", fontsize=10)
    ax2.grid(True, linestyle=':', alpha=0.6)
    
    # Annotate status
    if gm > 1.5:
        status_text = "Status: STIFF VESSEL\nHigh stability, but will snap back\nviolently in seaway."
        color_box = 'lightgreen'
    elif 0.3 <= gm <= 1.5:
        status_text = "Status: OPTIMAL TRACE\nGood operational balance\nbetween safety and comfort."
        color_box = 'honeydew'
    elif 0 < gm < 0.3:
        status_text = "Status: TENDER VESSEL\nSluggish roll response.\nDangerously low margin!"
        color_box = 'mistyrose'
    else:
        status_text = "🚨 CRITICAL FAILURE:\nNegative GM! Vessel will capsize\nspontaneously (Angle of Loll)."
        color_box = 'salmon'
        
    ax1.text(5, ax1.get_ylim()[0] + (ax1.get_ylim()[1]-ax1.get_ylim()[0])*0.7, status_text, 
             bbox=dict(boxstyle='round', facecolor=color_box, alpha=0.8), family='monospace', fontsize=9)
    
    plt.tight_layout()
    plt.show()

print("MANIPULATE HULL LOADING TO OBSERVE STABILITY CRITERIA:")
interact(simulate_stability_curve,
         vessel_weight_tonnes=widgets.FloatSlider(min=500.0, max=3000.0, step=100.0, value=1500.0, description='Displ (T):'),
         kg_meters=widgets.FloatSlider(min=4.0, max=7.0, step=0.1, value=5.2, description='KG Height (m):'));

MANIPULATE HULL LOADING TO OBSERVE STABILITY CRITERIA:


interactive(children=(FloatSlider(value=1500.0, description='Displ (T):', max=3000.0, min=500.0, step=100.0), …

---

## 2. Ballast Control and Load Conditions

To prevent a vessel from becoming too "stiff" or too "tender", engineers implement ballast control loops. Pumping seawater into low-lying tanks lowers the Center of Gravity ($KG \downarrow$), thereby widening the metacentric height ($GM \uparrow$).

Mathematically, if we distribute or shift a mass $m_{ballast}$ by a vertical distance $z_{shift}$, the new compound center of gravity is computed via standard static moments:

$$KG_{\text{new}} = \frac{(M_{\text{vessel}} \cdot KG_{\text{old}}) + (m_{\text{ballast}} \cdot z_{\text{tank}})}{M_{\text{vessel}} + m_{\text{ballast}}}$$

Let's map out how a real-time ballast correction routine shifts the system parameters back into a safe operating envelope after taking on top-heavy deck cargo.

In [2]:
def calculate_ballast_tuning(deck_cargo_tonnes, ballast_tank_fullness_pct):
    """
    Computes how shifting ballast water levels rectifies the hull's center of gravity
    after heavy components are added to the upper deck.
    """
    lightship_mass = 1000.0 * 1000.0  # 1,000 tonnes baseline
    lightship_kg = 5.0                # Baseline CG height
    km = 6.2                          # Metacenter height
    
    # Deck cargo added at the top superstructure (Z = 12.0 meters high)
    cargo_mass = deck_cargo_tonnes * 1000.0
    cargo_kg = 12.0
    
    # Ballast tanks located at the flat keel bottom (Z = 0.5 meters low)
    max_ballast_mass = 400.0 * 1000.0  # 400 tonnes max capacity
    ballast_mass = max_ballast_mass * (ballast_tank_fullness_pct / 100.0)
    ballast_kg = 0.5
    
    # Unified Composite Center of Gravity Equation
    total_mass = lightship_mass + cargo_mass + ballast_mass
    composite_kg = ((lightship_mass * lightship_kg) + (cargo_mass * cargo_kg) + (ballast_mass * ballast_kg)) / total_mass
    
    final_gm = km - composite_kg
    
    print("=" * 70)
    print("                 BALLAST CONTROL DECK MONITOR")
    print("=" * 70)
    print(f"  Total Displacement : {total_mass/1000:.1f} Tonnes")
    print(f"  Deck Cargo Weight  : {deck_cargo_tonnes:.1f} Tonnes (At Z=12.0m)")
    print(f"  Active Ballast     : {ballast_mass/1000:.1f} Tonnes (At Z=0.5m)")
    print("-" * 70)
    print(f"  Resulting KG Height: {composite_kg:.3f} meters above keel")
    print(f"  Calculated GM      : {final_gm:+.3f} meters")
    print("-" * 70)
    
    if final_gm >= 0.5:
        print("✅ STABILITY MONITOR: Within safe operating window.")
    elif 0.0 < final_gm < 0.5:
        print("⚠️ MARGINAL STATUS: Low righting lever. Interlocking thruster speed restrictions.")
    else:
        print("❌ INSTABILITY CRITICAL: Capsizing risk. Automatically engaging emergency ballast intake!")

interact(calculate_ballast_tuning,
         deck_cargo_tonnes=widgets.FloatSlider(min=0.0, max=300.0, step=20.0, value=180.0, description='Cargo (T):'),
         ballast_tank_fullness_pct=widgets.FloatSlider(min=0.0, max=100.0, step=5.0, value=30.0, description='Ballast %:'));

interactive(children=(FloatSlider(value=180.0, description='Cargo (T):', max=300.0, step=20.0), FloatSlider(va…

---

## 3. The Natural Period: The Bridge to Seakeeping

Hydrostatics doesn't just evaluate whether a ship will float upright—it acts as the **restoring spring constant** of the mechanical system. When displaced by an external perturbation (like a wave or wind load), the vessel will roll at its own characteristic **Natural Period ($T_{\phi}$)**.

Fossen derives this rotational period by balancing the rigid body roll inertia ($I_x$), the hydrodynamic roll added mass ($A_{44}$ from potential theory), and the hydrostatic stiffness ($mg \cdot GM$):

$$T_{\phi} = 2\pi \sqrt{\frac{I_x + A_{44}}{m g \cdot GM}}$$

This represents the crucial architectural handoff between **Chapter 4 (Statics)** and **Chapter 5 (Dynamic Seakeeping)**. If the incoming sea state has wave periods close to $T_{\phi}$, the system encounters **resonance**, generating catastrophic rolling amplitudes even in moderate conditions.

In [6]:
def visual_natural_period(gm_lever, added_mass_multiplier):
    """
    Simulates and plots the natural un-damped oscillation period of the hull.
    Demonstrates how hydrostatic tuning changes the frequency profile relative 
    to a FIXED ocean wave period.
    """
    m = 1200000.0  # 1,200 Tonnes
    g = 9.81
    beam = 12.0
    ix = (1.0 / 12.0) * m * (beam**2)
    a44 = ix * added_mass_multiplier
    
    total_inertia = ix + a44
    stiffness = m * g * gm_lever
    
    if stiffness <= 0:
        print("❌ Cannot compute oscillation period for an unstable hull configuration.")
        return
        
    omega_0 = np.sqrt(stiffness / total_inertia)
    t_period = (2.0 * np.pi) / omega_0
    
    t = np.linspace(0, 30, 300)
    roll_response = 10.0 * np.cos(omega_0 * t) 
    
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(t, roll_response, color='darkviolet', linewidth=2.5, label='Vessel Natural Roll Decay')
    
    # FIX: Lock the ocean wave energy boundary to a static window (e.g., 7.5 to 8.5 seconds)
    fixed_ocean_period = 8.0
    ax.axvspan(fixed_ocean_period - 0.5, fixed_ocean_period + 0.5, color='orange', alpha=0.25, label='Fixed Sea State Waves (8s Period)')
    
    ax.set_title(f"Hydrostatic Spring Response | Natural Period ($T_\phi$): {t_period:.2f} seconds", fontsize=12)
    ax.set_xlabel("Time Elapsed (Seconds)", fontsize=10)
    ax.set_ylabel("Roll Angle (Degrees)", fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper right')
    
    print("-" * 75)
    print(f"  Vessel Natural Period: {t_period:.2f} seconds")
    print(f"  Target Ocean Waves   : {fixed_ocean_period:.1f} seconds")
    print("-" * 75)
    
    # Calculate proximity to resonance
    if abs(t_period - fixed_ocean_period) < 0.6:
        print("🚨 CRITICAL RESONANCE DETECTED:")
        print("  Your ship's natural swing matches the ocean's wave rhythm!")
        print("  WAVE INJECTION RISK: Roll angles will amplify dangerously. Change ballast immediately.")
    else:
        print("✅ FREQUENCY DECOUPLED:")
        print("  The ship's natural period is safely offset from the dominant wave period.")

interact(visual_natural_period,
         gm_lever=widgets.FloatSlider(min=0.1, max=2.5, step=0.1, value=0.8, description='GM Lever (m):'),
         added_mass_multiplier=widgets.FloatSlider(min=0.1, max=1.0, step=0.05, value=0.4, description='A44 Fluid Multiplier:'));

<>:33: SyntaxWarning: invalid escape sequence '\p'
<>:33: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_46970/2745247647.py:33: SyntaxWarning: invalid escape sequence '\p'
  ax.set_title(f"Hydrostatic Spring Response | Natural Period ($T_\phi$): {t_period:.2f} seconds", fontsize=12)


interactive(children=(FloatSlider(value=0.8, description='GM Lever (m):', max=2.5, min=0.1), FloatSlider(value…

---

## Summary of Chapter 4 Physics

1. **Restoring Forces ($g(\eta)$)** act as the gravity-buoyancy spring configuration of the system, trying to force the vehicle back into neutral equilibrium.
2. **The Metacentric Height ($GM$)** dictates the stiffness of this spring. High $GM$ values yield quick, aggressive stability response profiles; low $GM$ values generate slow, sluggish, or unstable paths.
3. **Ballast Systems** rewrite the mass tracking layout in real time, moving the composite Center of Gravity ($KG$) to optimize stability bounds against changing operational environments or payloads.
4. **The Natural Period ($T_{\phi}$)** bridges hydrostatics to fluid seakeeping dynamics. Tuning $T_{\phi}$ via ballast allocation allows a vessel to actively decouple its resonance frequency from prevailing ocean wave spectra ($Chapter 5$).